In [25]:
import torch

In [52]:
tokens = ["Your", "journey", "starts", "with", "one", "step"]
x = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     
   [0.55, 0.87, 0.66], # journey  
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]] # step
)

In [60]:
#computing attension scores -> w1 = [ x1.x1, x1.x2, x1.x3, x1.x4, x1.x5, x1.x6 ] --> dot product between inputs 
#for each token, we are computing a vector that tells how close that token is with other tokens in the sequence (dot product = similarity distance)
# if you have n tokens in one input, then attension score size -> nxn
w = torch.empty(x.shape[0],x.shape[0])
print("Attension score")
print("         Your  Journey  starts    with    one    step ")
for i in range(0, x.shape[0]):
    for j in range(0, x.shape[0]):
        w[i][j] = x[i].dot(x[j])
    print(w[i], tokens[i])

Attension score
         Your  Journey  starts    with    one    step 
tensor([0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310]) Your
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]) journey
tensor([0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605]) starts
tensor([0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565]) with
tensor([0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935]) one
tensor([0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]) step


In [28]:
#computing attension weight by normalization for -> 1. sum of all to be 1 & always positive. higher weights higher importance
a = torch.softmax(w, dim=-1)
print(a)
print(a.sum(dim=-1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [29]:
#computing context vector zi -> ai x xi (attension weight of token x input token embed)
#working with just token 2
a2 = a[1]
print("attension weight = ",a2, "\n input token = ",x)
z2 = a2.matmul(x)
print("context vector =",z2)

attension weight =  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]) 
 input token =  tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
context vector = tensor([0.4419, 0.6515, 0.5683])


In [30]:
#context vectors -> weighted sum over inputs 
z = a.matmul(x)
print(z)


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [31]:
a2.shape, x.shape

(torch.Size([6]), torch.Size([6, 3]))

In [83]:
total = torch.zeros(3)
print("Calculation for query (1st token) Your:")
for i in range(0,6):
    val1 = x[i]
    val2 = a[0][i]
    print("Attention (Your,",tokens[i],")",val2," * ",tokens[i],"embed vector", val1)
    print("=",val1 * val2,". Attention tells us how close is Your to ",tokens[i],".After mul, we get weighted representation of",tokens[i],"w.r.t Your\n")
    temp = val1 * val2
    total = total + temp
print("weighted input vector with respect to Your = sum of all above outputs->",total)
print("\nsimilarly, weighted input vectors of all tokens are calculated -> all Context vectors Z")
for i in range(0,6):
    print(z[i]," -> weighted input vector with respect to", tokens[i])

Calculation for query (1st token) Your:
Attention (Your, Your ) tensor(0.2098)  *  Your embed vector tensor([0.4300, 0.1500, 0.8900])
= tensor([0.0902, 0.0315, 0.1868]) . Attention tells us how close is Your to  Your .After mul, we get weighted representation of Your w.r.t Your

Attention (Your, journey ) tensor(0.2006)  *  journey embed vector tensor([0.5500, 0.8700, 0.6600])
= tensor([0.1103, 0.1745, 0.1324]) . Attention tells us how close is Your to  journey .After mul, we get weighted representation of journey w.r.t Your

Attention (Your, starts ) tensor(0.1981)  *  starts embed vector tensor([0.5700, 0.8500, 0.6400])
= tensor([0.1129, 0.1684, 0.1268]) . Attention tells us how close is Your to  starts .After mul, we get weighted representation of starts w.r.t Your

Attention (Your, with ) tensor(0.1242)  *  with embed vector tensor([0.2200, 0.5800, 0.3300])
= tensor([0.0273, 0.0721, 0.0410]) . Attention tells us how close is Your to  with .After mul, we get weighted representation 